# Aggregations and Grouping with Dataframes

Using the F1 source data files in the folder "f1-sourcefiles/" create DataFrames.

This jupyter notebook covers:
- `groupBy()` + `agg()` with functions like `sum()`, `count()`, `avg()`, `max()`, `min()`
- `pivot()` for reshaping data
- Window functions (`ROW_NUMBER`, `RANK`, `LAG`, `LEAD`)


In [1]:
# Initalise a spark session
import os
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

# Fix JAVA_HOME to your actual Java 21 path
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PYSPARK_SUBMIT_ARGS"] = "--packages io.delta:delta-spark_2.12:3.2.0 pyspark-shell"

# Build Spark session with Delta Lake support
builder = SparkSession.builder \
    .appName("DeltaLakeExample") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

26/03/26 23:14:12 WARN Utils: Your hostname, DESKTOP-OQT8U26 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/26 23:14:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/robyip/projects/pyspark-deltalake/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/robyip/.ivy2/cache
The jars for the packages stored in: /home/robyip/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4e3dd4c4-5b49-4e2f-8dfc-8edb6aa4d189;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 172ms :: artifacts dl 9ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0 

In [2]:
# Load drivers.csv file
import time

start = time.time()

df_drivers = spark.read.csv("f1-sourcefiles/drivers.csv", header=True, inferSchema=True)

df_drivers.show()

end = time.time()

print(f"Time takes: {end - start:.2f} seconds")


26/03/26 23:14:28 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


+--------+----------+------+----+---------+----------+----------+-----------+--------------------+
|driverId| driverRef|number|code| forename|   surname|       dob|nationality|                 url|
+--------+----------+------+----+---------+----------+----------+-----------+--------------------+
|       1|  hamilton|    44| HAM|    Lewis|  Hamilton|1985-01-07|    British|http://en.wikiped...|
|       2|  heidfeld|    \N| HEI|     Nick|  Heidfeld|1977-05-10|     German|http://en.wikiped...|
|       3|   rosberg|     6| ROS|     Nico|   Rosberg|1985-06-27|     German|http://en.wikiped...|
|       4|    alonso|    14| ALO| Fernando|    Alonso|1981-07-29|    Spanish|http://en.wikiped...|
|       5|kovalainen|    \N| KOV|   Heikki|Kovalainen|1981-10-19|    Finnish|http://en.wikiped...|
|       6|  nakajima|    \N| NAK|   Kazuki|  Nakajima|1985-01-11|   Japanese|http://en.wikiped...|
|       7|  bourdais|    \N| BOU|Sébastien|  Bourdais|1979-02-28|     French|http://en.wikiped...|
|       8|

In [3]:
# GroupBy() 
df_drivers.groupBy("nationality").count().show()


+-----------------+-----+
|      nationality|count|
+-----------------+-----+
|          Mexican|    6|
|        Rhodesian|    4|
|          Finnish|    9|
|            Swiss|   23|
|             Thai|    2|
|           Indian|    2|
|    South African|   23|
|          Chinese|    1|
|       Indonesian|    1|
|Argentine-Italian|    1|
|          Chilean|    1|
|            Irish|    5|
|        Argentine|   24|
|           Polish|    1|
|          British|  166|
|         Japanese|   20|
|       Australian|   19|
|          Spanish|   15|
|       Portuguese|    4|
|        Hungarian|    1|
+-----------------+-----+
only showing top 20 rows



In [10]:
# .groupBy() - agg() with mean(), sum() count(), min() and max() by driver


from pyspark.sql import functions as F

df_results = spark.read.csv("f1-sourcefiles/results.csv", header=True, inferSchema=True)

df_results.groupBy("driverId").agg(
    F.min("positionOrder").alias("min_final_postiion"),
    F.max("positionOrder").alias("max_final_postiion"),
    F.mean("positionOrder").alias("mean_final_postiion"),
    F.sum("laps").alias("total_laps"),
    F.count("*").alias("total_races")
).show()


+--------+------------------+------------------+-------------------+----------+-----------+
|driverId|min_final_postiion|max_final_postiion|mean_final_postiion|total_laps|total_races|
+--------+------------------+------------------+-------------------+----------+-----------+
|     148|                 7|                38| 27.636363636363637|       299|         22|
|     463|                15|                29| 24.333333333333332|        64|          3|
|     471|                 9|                 9|                9.0|        76|          1|
|     496|                 3|                21|               10.5|       541|         12|
|     833|                12|                19| 15.692307692307692|       737|         13|
|     243|                 3|                33| 14.721311475409836|      2452|         61|
|     392|                11|                19|               15.0|        81|          2|
|     540|                19|                23|               21.0|        17| 

In [12]:
# Groupby() multiple columns


df_results.groupBy("driverId", "constructorId").agg(
    F.min("positionOrder").alias("min_final_postiion"),
    F.max("positionOrder").alias("max_final_postiion"),
    F.mean("positionOrder").alias("mean_final_postiion"),
    F.sum("laps").alias("total_laps"),
    F.count("*").alias("total_races")
).show()


+--------+-------------+------------------+------------------+-------------------+----------+-----------+
|driverId|constructorId|min_final_postiion|max_final_postiion|mean_final_postiion|total_laps|total_races|
+--------+-------------+------------------+------------------+-------------------+----------+-----------+
|      43|            7|                 6|                20| 12.107142857142858|      1484|         28|
|     158|           27|                 2|                29|               13.8|       648|         15|
|     408|           66|                 4|                23|               13.6|       154|          5|
|     485|          172|                28|                28|               28.0|         0|          1|
|     478|           66|                 9|                 9|                9.0|        36|          1|
|     554|          105|                 2|                23|               7.96|      1390|         25|
|     459|           87|                 8|   

In [ ]:
# groupBy() with a pivot() function



In [ ]:
# Windowing functions - (ROW_NUMBER, RANK, LAG, LEAD)

